# Chapitre 14 · Compter ce qui coûte : ta calculatrice de coûts

Notebook du chapitre 14 de *Construire un LLM de zéro* · Partie III « S'entraîner comme un labo ».

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et
exécutable de bout en bout. Lis, exécute, modifie pour voir. Tu y construis une
**calculatrice de coûts** (paramètres, mémoire, FLOPs, temps) validée sur le
**GPT du chapitre 10**. À la fin, la section **Exercices** : quatre défis à trous,
du plus simple au plus costaud, validés par des `assert`.

Tout tourne **hors ligne, sans GPU** ; les extrapolations vers un GPU sont
clairement étiquetées *estimation*.

## Setup

PyTorch, et rien d'autre. Tout le chapitre est du calcul et de la mesure sur CPU.

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print("PyTorch", torch.__version__)

## 1. Sortir la calculatrice avant de tourner la clé

Avant de tourner la clé, un labo **compte** : paramètres, mémoire, FLOPs, temps.
La question qui gouverne tout le chapitre : *est-ce que ça rentre dans le T4
gratuit de Colab, et ça prendra combien de temps ?* On y répond sur un modèle que
tu connais par cœur : le GPT du chapitre 10. Tes chiffres, mesurés.

## 2. Compter les paramètres, couche par couche

Un paramètre, c'est un nombre rangé dans une case ; les compter, c'est de
l'arithmétique de shapes : une matrice `(a, b)` en a `a * b`, un vecteur de
taille `d` en a `d`. Pour compter sur du réel, on rebâtit le GPT du chapitre 10 :
même configuration, même architecture. On ne l'entraîne pas ici, on le **compte**.

In [ ]:
# La configuration exacte du chapitre 10.
vocab_size = 81      # 81 caractères distincts dans les fables
block_size = 64      # contexte : jusqu'à 64 caractères
d_model    = 96      # largeur du modèle
n_heads    = 4       # têtes d'attention
n_layers   = 2       # blocs Transformer empilés
d_ff       = 384     # dimension cachée du FFN (4 x d_model)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        split = lambda t: t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        Q, K, V = split(self.W_Q(x)), split(self.W_K(x)), split(self.W_V(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask[:T, :T] == 0, float("-inf"))
        melange = torch.softmax(scores, dim=-1) @ V
        out = melange.transpose(1, 2).contiguous().view(B, T, d_model)
        return self.W_O(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.table_tokens = nn.Embedding(vocab_size, d_model)
        self.table_positions = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)
        self.register_buffer("masque", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T = x.shape
        h = self.table_tokens(x) + self.table_positions(torch.arange(T))
        for bloc in self.blocs:
            h = bloc(h, self.masque)
        return self.tete(self.ln_final(h))


torch.manual_seed(42)
gpt = GPT()
print("GPT du chapitre 10 reconstruit.")

Le décompte, étage par étage : les deux tables d'entrée, puis pour chaque bloc
l'attention (quatre matrices sans biais), le FFN (deux couches avec biais) et les
deux LayerNorm, puis la LayerNorm finale et la tête. `param_count` additionne le
tout et rend le détail.

In [ ]:
def param_count(vocab_size, block_size, d_model, n_heads, n_layers, d_ff):
    """Compte les parametres d'un GPT decodeur, couche par couche (a la main)."""
    tok_emb = vocab_size * d_model            # table des tokens (81 x 96)
    pos_emb = block_size * d_model            # table des positions (64 x 96)

    # un bloc Transformer :
    attn = 4 * (d_model * d_model)            # W_Q, W_K, W_V, W_O, sans biais
    ln = 2 * d_model                          # gamma + beta d'une LayerNorm
    ffn = (d_model * d_ff + d_ff) + (d_ff * d_model + d_model)  # W1+b1, W2+b2
    bloc = attn + ffn + 2 * ln                # deux LayerNorm par bloc

    ln_final = 2 * d_model                    # LayerNorm finale
    tete = d_model * vocab_size               # tete de sortie, sans biais

    total = tok_emb + pos_emb + n_layers * bloc + ln_final + tete
    return {
        "tokens": tok_emb, "positions": pos_emb,
        "attention/bloc": attn, "ffn/bloc": ffn, "layernorms/bloc": 2 * ln,
        "bloc": bloc, "ln_final": ln_final, "tete": tete, "total": total,
    }


detail = param_count(vocab_size, block_size, d_model, n_heads, n_layers, d_ff)
for k, v in detail.items():
    print(f"{k:>18} : {v:>9,}".replace(",", " "))

Maintenant le geste qui vaut de l'or : on **vérifie** notre addition contre
PyTorch. `p.numel()` (*number of elements*) rend le nombre de nombres d'un
tenseur ; on somme sur tous les paramètres du modèle. Si les deux coïncident,
tu as compris l'architecture.

In [ ]:
total = sum(p.numel() for p in gpt.parameters())
print(total)           # 244800

a_la_main = detail["total"]
print(f"a la main : {a_la_main:>9,}".replace(",", " "))
assert a_la_main == total == 244_800, "le compte a la main doit coller a numel"
print("\nLe decompte a la main colle a sum(p.numel()). 244 800 parametres.")

## 3. La mémoire : combien d'octets par paramètre ?

Un paramètre à entraîner, ce n'est pas un nombre en mémoire, c'est **quatre** :
le poids, son gradient, et les deux compteurs d'AdamW (moyenne et variance).
En fp32, chacun pèse 4 octets, soit **16 octets par paramètre**. La précision
mixte (bf16) ne gagne pas sur l'optimiseur : elle garde poids de référence et
états d'AdamW en fp32. Son gain est sur la vitesse et sur les **activations**.

In [ ]:
def memoire_entrainement(n_params, octets_par_param=16):
    """Etat d'entrainement (poids + gradient + m + v d'AdamW), en octets.
    16 octets/param en fp32 ; ~16 aussi en precision mixte classique."""
    return n_params * octets_par_param


octets = memoire_entrainement(244_800)
print(f"Notre GPT : {octets:,} octets".replace(",", " "), f"= {octets/1e6:.2f} Mo")

# la meme regle a l'echelle du milliard de parametres
un_milliard = memoire_entrainement(1_000_000_000)
print(f"1 milliard de params : {un_milliard/1e9:.1f} Go (l'etat, avant activations)")
print("-> 16 Go, soit un T4 entier, rien que pour 1 milliard de parametres.")

Les **activations** sont le poste qu'on oublie. Pendant le forward, chaque couche
produit des tenseurs intermédiaires qu'il faut garder jusqu'au backward. Elles
gonflent avec le **batch**, la **longueur de séquence** et le **nombre de
couches**, pas avec le seul modèle. Ordre de grandeur (facteur `k` ~ 10) :

In [ ]:
def activations_approx(batch, seq, d_model, n_layers, k=10, octets=4):
    """Estimation grossiere des activations gardees pour le backward, en octets."""
    return batch * seq * d_model * n_layers * k * octets


for batch in (32, 128):
    a = activations_approx(batch, block_size, d_model, n_layers)
    print(f"batch={batch:>3} : ~{a/1e6:.2f} Mo d'activations (fp32)")
print("\nRetiens : l'etat de l'optimiseur ne depend que du modele ;")
print("les activations, elles, explosent avec le batch et le contexte.")

## 4. Compter le calcul : la règle des 6·N·D

Le compute total d'un entraînement, en **FLOPs**, tient en une multiplication :
`C ≈ 6 · N · D`. Le 6 vient de deux passes par token : ~2N FLOPs pour l'avant
(une multiplication + une addition par paramètre) et ~4N pour l'arrière (deux
gradients par paramètre). 2N + 4N = 6N par token, × D tokens vus.

In [ ]:
def flop_count(n_params: int, n_tokens: int) -> int:
    """C ≈ 6·N·D, le compute total d'un entraînement en FLOPs."""
    return 6 * n_params * n_tokens


# Les chiffres reels de ton run du chapitre 10 : 3000 pas, batch 32, seq 64.
n_steps = 3000
batch = 32
D = n_steps * batch * block_size
print(f"D = {n_steps} pas x {batch} x {block_size} = {D:,} tokens vus".replace(",", " "))

C = flop_count(244_800, D)
print(f"C = 6 x 244 800 x {D:,} = {C:,} FLOPs".replace(",", " "))
print(f"  = {C:.2e} FLOPs (~9 teraFLOPs)")

# La meme multiplication a l'echelle de GPT-3 (estimation, pas une mesure locale).
C_gpt3 = flop_count(175_000_000_000, 300_000_000_000)
print(f"\nGPT-3 (estimation) : {C_gpt3:.2e} FLOPs, soit ~{C_gpt3/C:.1e}x notre run.")

## 5. Combien de temps ? Le débit du GPU et le MFU

Le temps, c'est le calcul divisé par le **débit réel**. Le débit réel n'est pas le
chiffre du constructeur : c'est le débit crête × **MFU** (*Model FLOPs
Utilization*), la part de la puissance que ton code exploite vraiment (souvent
0,3 à 0,5). On fait d'abord l'**erreur naïve** (MFU = 100 %), puis on la
confronte à une **mesure**.

In [ ]:
T4_CRETE = 65e12   # ~65 TFLOPs/s crete, calcul 16 bits, chiffre constructeur

# Estimation NAIVE : on suppose le GPU exploite a 100 %. C'est faux, on le montre.
t_naif = C / T4_CRETE
print(f"[estimation naive, MFU=100%] : {t_naif:.3f} s   <- trop beau pour etre vrai")


def temps_estime(n_params, n_tokens, debit_crete_flops, mfu):
    """Temps d'entraînement estimé, en secondes. mfu entre 0 et 1."""
    compute = flop_count(n_params, n_tokens)       # C = 6·N·D
    debit_reel = debit_crete_flops * mfu           # ce que le GPU sort vraiment
    return compute / debit_reel


for mfu in (0.3, 0.5):
    t = temps_estime(244_800, D, T4_CRETE, mfu)
    print(f"[estimation T4, MFU={mfu:.0%}] : {t:.3f} s")

Maintenant la **mesure** : on chronomètre un vrai pas d'entraînement de ton GPT,
ici sur CPU. On en déduit le débit effectif. (Sur ton portable, le temps par pas
variera ; c'est normal, c'est ta mesure à toi.)

In [ ]:
torch.manual_seed(42)
gpt_train = GPT()
opt = torch.optim.AdamW(gpt_train.parameters(), lr=3e-3)
faux_data = torch.randint(0, vocab_size, (20_000,))


def un_batch(taille=32):
    ix = torch.randint(0, len(faux_data) - block_size - 1, (taille,))
    x = torch.stack([faux_data[i : i + block_size] for i in ix])
    y = torch.stack([faux_data[i + 1 : i + block_size + 1] for i in ix])
    return x, y


def un_pas():
    x, y = un_batch()
    loss = F.cross_entropy(gpt_train(x).view(-1, vocab_size), y.view(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()


for _ in range(5):      # echauffement
    un_pas()

debut = time.time()
K = 30
for _ in range(K):
    un_pas()
dt = (time.time() - debut) / K

tokens_par_pas = batch * block_size
debit_mesure = tokens_par_pas / dt
flops_mesure = flop_count(244_800, tokens_par_pas) / dt
mfu_cpu = flops_mesure / T4_CRETE   # a titre indicatif : un CPU n'est pas un T4

print(f"temps/pas mesure (CPU) : {dt*1000:.1f} ms")
print(f"debit mesure           : {debit_mesure:,.0f} tokens/s".replace(",", " "))
print(f"debit FLOPs mesure     : {flops_mesure:.2e} FLOPs/s")
print(f"'MFU' vs crete T4      : {mfu_cpu:.2%} (mesure CPU, comparaison indicative)")
print(f"\nrun complet 3000 pas (estimation) : ~{3000*dt:.0f} s")
print("(coherent avec les ~52 s du chapitre 10 sur une puce Apple)")

Le verdict : la promesse naïve (~0,14 s) est démentie par la mesure (~1 minute).
Le coupable, c'est le MFU qu'on avait ignoré. **Un chiffre mesuré vaut mille
chiffres de brochure.**

### Traduire les heures en FCFA

Le temps devient un prix dès qu'on loue le GPU à l'heure. Un coût en FCFA parle
plus fort qu'un coût en dollars quand c'est ton budget.

In [ ]:
USD_PAR_FCFA = 600     # 1 dollar ~ 600 FCFA
T4_USD_HEURE = 0.35     # ordre de grandeur d'un T4 loue a l'heure


def cout(temps_s, usd_heure=T4_USD_HEURE):
    heures = temps_s / 3600
    usd = heures * usd_heure
    return usd, usd * USD_PAR_FCFA


# Notre run, avec une estimation T4 a MFU=40 %.
t_run = temps_estime(244_800, D, T4_CRETE, mfu=0.4)
usd, fcfa = cout(t_run)
print(f"Notre GPT (estimation T4) : {t_run:.2f} s -> {usd:.5f} $ ~ {fcfa:.2f} FCFA")
print("Le prix d'un bonbon.\n")

# Un modele d'un milliard de params sur 20 milliards de tokens.
C_gros = flop_count(1_000_000_000, 20_000_000_000)
t_gros = C_gros / (T4_CRETE * 0.4)
usd_g, fcfa_g = cout(t_gros, usd_heure=2.0)   # sur un GPU plus costaud loue ~2 $/h
print(f"1 Md params x 20 Md tokens (estimation) : ~{t_gros/3600:.0f} h GPU")
print(f"  -> ~{usd_g:,.0f} $ ~ {fcfa_g:,.0f} FCFA".replace(",", " "))
print("La meme calculatrice, du bonbon au budget.")

## 6. Est-ce que ça rentre dans Colab ?

La question d'ingénieur avant de lancer. On compare l'état d'entraînement
(`N × 16` octets) au plafond du T4. **Un OOM se calcule avant de le subir.**
Le cas de GPT-2 XL déborde : on l'imprime, sans jamais rien allouer.

In [ ]:
T4_GO = 16.0   # memoire d'un T4 gratuit


def rentre_dans_t4(n_params, plafond_go=T4_GO):
    """Verdict memoire, calcule AVANT de lancer. N'alloue rien."""
    besoin_go = memoire_entrainement(n_params) / 1e9
    ok = besoin_go <= plafond_go
    if ok:
        verdict = f"RENTRE ({besoin_go:.2f} Go sur {plafond_go:.0f})"
    else:
        verdict = f"NE RENTRE PAS (depassement de {besoin_go - plafond_go:.1f} Go)"
    return ok, besoin_go, verdict


modeles = [
    ("notre GPT",     244_800),
    ("GPT-2 small",   124_000_000),
    ("GPT-2 large",   774_000_000),
    ("GPT-2 XL",      1_500_000_000),
]
for nom, n in modeles:
    ok, go, verdict = rentre_dans_t4(n)
    marque = "OK " if ok else "!! "
    print(f"{marque}{nom:<14} {go:>6.2f} Go  -> {verdict}")

Le cas qui échoue, exécuté **sans faire planter le notebook** : on tente
symboliquement d'entraîner GPT-2 XL sur un T4, et la calculatrice refuse en
amont. C'est l'`OutOfMemoryError` de Colab, mais calculé, pas subi.

In [ ]:
def preparer_entrainement(nom, n_params, plafond_go=T4_GO):
    ok, go, verdict = rentre_dans_t4(n_params, plafond_go)
    if not ok:
        raise MemoryError(f"{nom} : {verdict}")
    return f"{nom} : {verdict}, on peut lancer."


try:
    print(preparer_entrainement("GPT-2 XL", 1_500_000_000))
except MemoryError as e:
    print("Refuse AVANT de lancer :", e)
    print("-> on n'a alloue AUCUN tenseur. Trois multiplications, zero session gachee.")

# Ce qui, lui, passe :
print(preparer_entrainement("notre GPT", 244_800))

### Un aperçu : combien de données par paramètre ?

Si on te fixe un budget de calcul, comment le dépenser ? La règle empirique dite
**de Chinchilla** vise ~20 tokens de données par paramètre. L'histoire complète
(Kaplan, Chinchilla, l'overtraining moderne) est au **chapitre 15**.

In [ ]:
def tokens_chinchilla(n_params, ratio=20):
    """Nombre de tokens vise par la regle des ~20 tokens/parametre."""
    return ratio * n_params


for nom, n in [("notre GPT", 244_800), ("un 1 Md", 1_000_000_000)]:
    d = tokens_chinchilla(n)
    print(f"{nom:<12} ({n:,} params)".replace(",", " "),
          f"-> ~{d:,} tokens vises".replace(",", " "))
print("\nSuite au chapitre 15 : pourquoi les labos depassent volontairement 20:1.")

## Exercices

À toi de jouer : quatre exercices, du plus simple (●) au plus costaud (●●●).
Chaque cellule marquée `# TODO(toi)` contient un trou ; complète-le, puis exécute
la cellule de validation (`assert`) qui suit : si elle passe sans erreur, c'est
gagné. Tu reconstruiras toi-même, de mémoire, les cinq fonctions de la
calculatrice de coûts.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta
place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi.
Les réponses sont dans le notebook solution, à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · La mémoire et les 6·N·D — niveau ●

Deux règles de poche, deux fonctions. `memoire_entrainement` : un paramètre à
entraîner réclame 4 nombres (poids, gradient, moyenne et variance d'AdamW), à
4 octets chacun en fp32, soit 16 octets par paramètre. `flop_count` : la règle
reine du compute, `C ≈ 6 · N · D`.

In [ ]:
def memoire_entrainement(n_params, octets_par_param=16):
    """Etat d'entrainement en octets : 16 octets/param en fp32."""
    # TODO(toi) : multiplie le nombre de parametres par les octets par parametre
    return ...


def flop_count(n_params, n_tokens):
    """C ~ 6 * N * D, le compute total en FLOPs."""
    # TODO(toi) : applique la regle des 6*N*D
    return ...

In [ ]:
# Validation : mémoire et 6·N·D.
octets = memoire_entrainement(244_800)
assert octets is not ..., "remplace le ... par ton calcul"
assert octets == 3_916_800, f"244 800 x 16 = 3 916 800 octets, pas {octets}"
assert memoire_entrainement(1_000_000_000) == 16_000_000_000, "1 Md de params -> 16 Go"

n_steps, batch = 3000, 32
D = n_steps * batch * block_size                      # tokens vus pendant le run
C = flop_count(244_800, D)
assert C is not ..., "remplace le ... par ton calcul"
assert C == 9_024_307_200_000, f"C attendu 9.02e12, obtenu {C}"
print(f"Memoire : {octets/1e6:.2f} Mo | D = {D:,} tokens | C = {C:.2e} FLOPs".replace(",", " "))
print("Exercice 1 valide.")

### Exercice 2 · La calculatrice de temps — niveau ●●

Le temps réel = compute ÷ (débit crête × MFU). Complète `temps_estime` en trois
lignes : le compute avec `flop_count` (exercice 1), le débit réel en dégradant le
crête par le MFU, puis la division. La validation confronte l'erreur naïve
(MFU = 100 %) à l'estimation réaliste.

In [ ]:
def temps_estime(n_params, n_tokens, debit_crete_flops, mfu):
    """Temps d'entrainement estime, en secondes (mfu entre 0 et 1)."""
    # TODO(toi) : compute (6*N*D), puis debit reel (crete x MFU), puis compute / debit reel
    return ...

In [ ]:
# Validation : la calculatrice de temps.
n_steps, batch = 3000, 32
D = n_steps * batch * block_size
t_naif = flop_count(244_800, D) / T4_CRETE   # MFU = 100 %, l'erreur classique
print(f"[naif, MFU=100%] {t_naif:.3f} s  (trop beau pour etre vrai)")

t30 = temps_estime(244_800, D, T4_CRETE, 0.3)
assert t30 is not ..., "remplace le ... par ton code"
assert abs(t30 - t_naif / 0.3) < 1e-6, "temps = compute / (crete * mfu)"
print(f"[T4, MFU=30%] {t30:.3f} s")
print("Exercice 2 valide.")

### Exercice 3 · Le verdict « ça rentre dans le T4 ? » — niveau ●●

Complète `rentre_dans_t4` (version compacte : elle renvoie le couple
`(ok, besoin_go)`). On compare l'état d'entraînement (`N × 16` octets, via
`memoire_entrainement` de l'exercice 1) au plafond du T4. Le cas de GPT-2 XL
doit **déborder**, et on le montre sans jamais rien allouer : un OOM calculé,
pas subi.

In [ ]:
T4_GO = 16.0


def rentre_dans_t4(n_params, plafond_go=T4_GO):
    """Verdict memoire, calcule AVANT de lancer. Renvoie (ok, besoin_go)."""
    # TODO(toi) : besoin_go en Go (memoire_entrainement / 1e9),
    # ok si besoin_go <= plafond_go, renvoie (ok, besoin_go)
    return ...

In [ ]:
# Validation : le verdict mémoire, calculé avant de lancer.
for nom, n in [("notre GPT", 244_800), ("GPT-2 small", 124_000_000),
               ("GPT-2 large", 774_000_000), ("GPT-2 XL", 1_500_000_000)]:
    res = rentre_dans_t4(n)
    assert res is not ..., "remplace le ... par ton code"
    ok, go = res
    print(f"{'OK ' if ok else '!! '}{nom:<14} {go:>6.2f} Go -> {'rentre' if ok else 'NE RENTRE PAS'}")

ok_gpt, _ = rentre_dans_t4(244_800)
ok_xl, go_xl = rentre_dans_t4(1_500_000_000)
assert ok_gpt is True, "notre GPT tient largement"
assert ok_xl is False and abs(go_xl - 24.0) < 0.1, "GPT-2 XL doit deborder (24 Go)"

# Le cas qui echoue, montre proprement (try/except, aucune allocation).
try:
    ok, go = rentre_dans_t4(1_500_000_000)
    if not ok:
        raise MemoryError(f"GPT-2 XL : {go:.1f} Go demandes pour 16 disponibles")
except MemoryError as e:
    print("\nRefuse AVANT de lancer :", e)
    print("-> aucun tenseur alloue. Trois multiplications, zero session gachee.")
print("Exercice 3 valide.")

### Exercice 4 · Compter les paramètres à la main — niveau ●●●

Le grand décompte, sans regarder la section 2. Écris `param_count`, qui renvoie
le **total** (un entier). Rappel : une matrice `(a, b)` a `a*b` paramètres, un
vecteur de taille `d` en a `d`. Les `nn.Linear` du FFN ont un biais ; l'attention
et la tête n'en ont pas ; chaque LayerNorm a deux vecteurs appris.

In [ ]:
def param_count(vocab_size, block_size, d_model, n_heads, n_layers, d_ff):
    """Compte les parametres d'un GPT decodeur, couche par couche."""
    # TODO(toi) : les deux tables d'entree (tokens, positions)
    #   + n_layers blocs (attention sans biais, FFN avec biais, deux LayerNorm)
    #   + LayerNorm finale + tete sans biais
    total = ...
    return total

In [ ]:
# Validation : le décompte à la main contre numel.
a_la_main = param_count(vocab_size, block_size, d_model, n_heads, n_layers, d_ff)
mesure = sum(p.numel() for p in gpt.parameters())
assert a_la_main is not ..., "remplace le ... par ton calcul"
assert a_la_main == mesure == 244_800, (
    f"a la main = {a_la_main}, numel = {mesure} : les deux doivent valoir 244 800"
)
print("Exercice 4 valide : ton decompte a la main colle a numel. 244 800 parametres.")

## Verdict

Quatre validations vertes : tu as reconstruit de mémoire `memoire_entrainement`,
`flop_count`, `temps_estime`, `rentre_dans_t4` et `param_count`. Cinq fonctions,
et tu réponds à *ça rentre, ça coûte combien, ça prend combien de temps* pour
n'importe quel modèle, avant de lancer.

Si un `assert` bloque, les corrigés sont dans le notebook **solution**, à ouvrir
après avoir vraiment essayé. Suite au chapitre 15 : plusieurs GPU, et les lois
d'échelle en entier.